# Advanced RAG Patterns

## Graph RAG, Agentic RAG, and Advanced Retrieval Techniques

In this advanced notebook, you will learn:
1. **Graph RAG** - Knowledge graphs for relationship reasoning
2. **Agentic RAG** - AI agents that decide retrieval strategies
3. **HyDE** - Hypothetical Document Embeddings
4. **Query Expansion** - Multi-query retrieval
5. **Parent Document Retriever** - Hierarchical retrieval

**Prerequisites:**
- Complete Notebook 02 (Production RAG System)
- Understanding of basic RAG concepts

**Estimated Time:** 3-4 hours

## Part 1: Graph RAG - Knowledge Graph Enhanced Retrieval

In [ ]:
!pip install networkx pyvis
!pip install rdflib SPARQLWrapper
!pip install langchain-community

In [ ]:
import networkx as nx
from pyvis.network import Network
from typing import List, Dict, Tuple, Set
from dataclasses import dataclass
import json

print("✓ Graph RAG libraries loaded")

### Building a Knowledge Graph for RAG

In [ ]:
@dataclass
class KnowledgeTriple:
    """A subject-predicate-object triple"""
    subject: str
    predicate: str
    object: str
    
    def __str__(self):
        return f"({self.subject}) --[{self.predicate}]--> ({self.object})"


class KnowledgeGraph:
    """
    Knowledge Graph for RAG
    - Stores entities and relationships
    - Enables multi-hop reasoning
    - Complements vector search
    """
    
    def __init__(self):
        self.graph = nx.DiGraph()
        self.entity_index = {}  # Map entity to node id
    
    def add_triple(self, triple: KnowledgeTriple):
        """Add a knowledge triple to the graph"""
        subject = triple.subject
        predicate = triple.predicate
        obj = triple.object
        
        # Add nodes if they don't exist
        if subject not in self.entity_index:
            self.graph.add_node(subject, type='entity')
            self.entity_index[subject] = subject
        
        if obj not in self.entity_index:
            self.graph.add_node(obj, type='entity')
            self.entity_index[obj] = obj
        
        # Add edge
        self.graph.add_edge(subject, obj, relation=predicate)
    
    def add_triples(self, triples: List[KnowledgeTriple]):
        """Add multiple triples"""
        for triple in triples:
            self.add_triple(triple)
    
    def query_direct(self, entity: str) -> List[Tuple[str, str]]:
        """Get all direct relationships for an entity"""
        if entity not in self.graph:
            return []
        
        relationships = []
        
        # Outgoing edges
        for neighbor in self.graph.successors(entity):
            relation = self.graph[entity][neighbor]['relation']
            relationships.append((relation, neighbor))
        
        # Incoming edges
        for neighbor in self.graph.predecessors(entity):
            relation = self.graph[neighbor][entity]['relation']
            relationships.append((f"<--{relation}", neighbor))
        
        return relationships
    
    def query_multi_hop(self, start_entity: str, max_hops: int = 2) -> Set[str]:
        """Get all entities reachable within max_hops"""
        if start_entity not in self.graph:
            return set()
        
        visited = {start_entity}
        current_level = {start_entity}
        
        for _ in range(max_hops):
            next_level = set()
            for entity in current_level:
                # Successors
                for neighbor in self.graph.successors(entity):
                    if neighbor not in visited:
                        next_level.add(neighbor)
                        visited.add(neighbor)
                
                # Predecessors
                for neighbor in self.graph.predecessors(entity):
                    if neighbor not in visited:
                        next_level.add(neighbor)
                        visited.add(neighbor)
            
            current_level = next_level
        
        return visited - {start_entity}
    
    def find_path(self, source: str, target: str) -> List[str]:
        """Find shortest path between two entities"""
        try:
            path = nx.shortest_path(self.graph, source, target)
            return path
        except nx.NetworkXNoPath:
            return []
    
    def visualize(self, output_file: str = "knowledge_graph.html"):
        """Visualize the knowledge graph"""
        net = Network(height="600px", width="100%", bgcolor="#222222", font_color="white")
        
        # Add nodes
        for node in self.graph.nodes():
            net.add_node(node, label=node, title=node, color="#00ff00")
        
        # Add edges
        for edge in self.graph.edges(data=True):
            net.add_edge(edge[0], edge[1], title=edge[1]['relation'], label=edge[1]['relation'])
        
        net.show_buttons(filter_=['physics'])
        net.save_graph(output_file)
        print(f"Graph saved to {output_file}")


# Example: Build ML/AI knowledge graph
kg = KnowledgeGraph()

triples = [
    KnowledgeTriple("Machine Learning", "is_subset_of", "Artificial Intelligence"),
    KnowledgeTriple("Deep Learning", "is_subset_of", "Machine Learning"),
    KnowledgeTriple("Neural Networks", "used_in", "Deep Learning"),
    KnowledgeTriple("Transformers", "type_of", "Neural Networks"),
    KnowledgeTriple("BERT", "instance_of", "Transformers"),
    KnowledgeTriple("GPT", "instance_of", "Transformers"),
    KnowledgeTriple("Computer Vision", "uses", "CNN"),
    KnowledgeTriple("CNN", "type_of", "Neural Networks"),
    KnowledgeTriple("Reinforcement Learning", "type_of", "Machine Learning"),
    KnowledgeTriple("Q-Learning", "algorithm_in", "Reinforcement Learning"),
    KnowledgeTriple("AlphaGo", "uses", "Reinforcement Learning"),
    KnowledgeTriple("NLP", "application_of", "Transformers"),
    KnowledgeTriple("Sentiment Analysis", "task_in", "NLP"),
    KnowledgeTriple("Named Entity Recognition", "task_in", "NLP"),
]

kg.add_triples(triples)
print(f"✓ Knowledge graph built with {len(kg.graph.nodes())} entities and {len(kg.graph.edges())} relationships")

In [ ]:
# Query the knowledge graph
print("Direct relationships for 'Transformers':")
relationships = kg.query_direct("Transformers")
for relation, entity in relationships:
    print(f"  {relation} → {entity}")

print("\nMulti-hop relationships from 'Machine Learning' (2 hops):")
related = kg.query_multi_hop("Machine Learning", max_hops=2)
for entity in related:
    print(f"  - {entity}")

print("\nPath from 'BERT' to 'Artificial Intelligence':")
path = kg.find_path("BERT", "Artificial Intelligence")
print(f"  {' → '.join(path)}")

# Visualize
kg.visualize("ml_knowledge_graph.html")

### Graph RAG: Combining Vector Search with Knowledge Graph

In [ ]:
class GraphRAG:
    """
    Graph-enhanced RAG system
    - Uses vector search for semantic retrieval
    - Uses knowledge graph for relationship reasoning
    - Combines both for better answers
    """
    
    def __init__(self, knowledge_graph: KnowledgeGraph, vector_store, embedder):
        self.kg = knowledge_graph
        self.vector_store = vector_store
        self.embedder = embedder
    
    def query(self, question: str, use_graph: bool = True, 
              use_vector: bool = True, top_k: int = 5) -> Dict:
        """
        Query with both graph and vector retrieval
        
        Returns:
            Dict with retrieved information from both sources
        """
        results = {
            'question': question,
            'graph_results': [],
            'vector_results': [],
            'combined_context': ''
        }
        
        # Extract entities from question (simple keyword extraction)
        entities = self._extract_entities(question)
        
        if use_graph:
            # Graph-based retrieval
            for entity in entities:
                if entity in self.kg.entity_index:
                    # Get direct relationships
                    relationships = self.kg.query_direct(entity)
                    for relation, related_entity in relationships:
                        results['graph_results'].append({
                            'entity': entity,
                            'relation': relation,
                            'related': related_entity
                        })
                    
                    # Multi-hop reasoning
                    multi_hop = self.kg.query_multi_hop(entity, max_hops=2)
                    for related in multi_hop:
                        results['graph_results'].append({
                            'entity': entity,
                            'relation': 'multi_hop',
                            'related': related
                        })
        
        if use_vector:
            # Vector-based retrieval
            query_embedding = self.embedder.embed_query(question)
            # In production: vector_store.similarity_search(query_embedding, top_k)
            results['vector_results'] = []  # Placeholder
        
        # Combine results
        results['combined_context'] = self._build_context(results)
        
        return results
    
    def _extract_entities(self, text: str) -> List[str]:
        """Simple entity extraction (capitalize words)"""
        # In production, use NER model
        words = text.split()
        entities = [word for word in words if word[0].isupper()]
        return entities
    
    def _build_context(self, results: Dict) -> str:
        """Build context from retrieved information"""
        context_parts = []
        
        # Add graph information
        if results['graph_results']:
            graph_text = "Knowledge Graph Information:\n"
            for item in results['graph_results']:
                graph_text += f"- {item['entity']} {item['relation']} {item['related']}\n"
            context_parts.append(graph_text)
        
        # Add vector information
        if results['vector_results']:
            vector_text = "Document Information:\n"
            for doc in results['vector_results']:
                vector_text += f"- {doc}\n"
            context_parts.append(vector_text)
        
        return "\n".join(context_parts)


# Example usage
print("Graph RAG Example")
print("=" * 60)

# Create mock vector store
class MockVectorStore:
    def similarity_search(self, query, top_k):
        return []

class MockEmbedder:
    def embed_query(self, text):
        return None

graph_rag = GraphRAG(kg, MockVectorStore(), MockEmbedder())

# Query
question = "What is the relationship between BERT and Artificial Intelligence?"
results = graph_rag.query(question)

print(f"\nQuestion: {question}")
print(f"\nRetrieved context:")
print(results['combined_context'])

## Part 2: Agentic RAG - AI Agents for Intelligent Retrieval

In [ ]:
from enum import Enum
from typing import Optional

class RetrievalStrategy(Enum):
    DIRECT = "direct"  # Simple vector search
    EXPANDED = "expanded"  # Query expansion
    HYPOTHETICAL = "hypothetical"  # HyDE
    GRAPH = "graph"  # Graph-based
    MULTI_HOP = "multi_hop"  # Multi-step retrieval


class RetrievalAgent:
    """
    AI Agent that decides retrieval strategy
    - Analyzes the query
    - Chooses appropriate retrieval strategy
    - Executes retrieval
    - Synthesizes answer
    """
    
    def __init__(self, llm=None):
        self.llm = llm
        self.strategy_history = []
    
    def analyze_query(self, query: str) -> Dict:
        """
        Analyze query to determine retrieval strategy
        
        Features to consider:
        - Query length
        - Question type (what, how, why, compare)
        - Complexity (single entity vs multi-entity)
        - Domain specificity
        """
        analysis = {
            'query': query,
            'length': len(query),
            'question_type': self._detect_question_type(query),
            'num_entities': self._count_entities(query),
            'complexity': self._estimate_complexity(query),
            'recommended_strategy': None
        }
        
        # Rule-based strategy selection
        if analysis['num_entities'] > 1:
            analysis['recommended_strategy'] = RetrievalStrategy.EXPANDED
        elif analysis['question_type'] in ['why', 'how']:
            analysis['recommended_strategy'] = RetrievalStrategy.HYPOTHETICAL
        elif analysis['complexity'] == 'high':
            analysis['recommended_strategy'] = RetrievalStrategy.MULTI_HOP
        else:
            analysis['recommended_strategy'] = RetrievalStrategy.DIRECT
        
        return analysis
    
    def _detect_question_type(self, query: str) -> str:
        """Detect question type"""
        query_lower = query.lower()
        
        if query_lower.startswith('what'):
            return 'what'
        elif query_lower.startswith('how'):
            return 'how'
        elif query_lower.startswith('why'):
            return 'why'
        elif query_lower.startswith('compare'):
            return 'compare'
        elif query_lower.startswith('list') or query_lower.startswith('name'):
            return 'list'
        else:
            return 'other'
    
    def _count_entities(self, query: str) -> int:
        """Count named entities (simple heuristic)"""
        words = query.split()
        capitalized = [w for w in words if w[0].isupper() and len(w) > 1]
        return len(capitalized)
    
    def _estimate_complexity(self, query: str) -> str:
        """Estimate query complexity"""
        if len(query.split()) > 15:
            return 'high'
        elif len(query.split()) > 8:
            return 'medium'
        else:
            return 'low'
    
    def execute_retrieval(self, query: str, strategy: RetrievalStrategy) -> Dict:
        """Execute retrieval with chosen strategy"""
        
        self.strategy_history.append({
            'query': query,
            'strategy': strategy.value
        })
        
        if strategy == RetrievalStrategy.DIRECT:
            return self._direct_retrieval(query)
        elif strategy == RetrievalStrategy.EXPANDED:
            return self._expanded_retrieval(query)
        elif strategy == RetrievalStrategy.HYPOTHETICAL:
            return self._hyde_retrieval(query)
        elif strategy == RetrievalStrategy.GRAPH:
            return self._graph_retrieval(query)
        elif strategy == RetrievalStrategy.MULTI_HOP:
            return self._multi_hop_retrieval(query)
        else:
            return {'error': 'Unknown strategy'}
    
    def _direct_retrieval(self, query: str) -> Dict:
        """Simple vector search"""
        return {'strategy': 'direct', 'results': [], 'query': query}
    
    def _expanded_retrieval(self, query: str) -> Dict:
        """Query expansion: generate multiple queries"""
        # In production: use LLM to generate variations
        expanded_queries = [
            query,
            f"Explain {query}",
            f"What do you know about {query}"
        ]
        return {
            'strategy': 'expanded',
            'original_query': query,
            'expanded_queries': expanded_queries,
            'results': []
        }
    
    def _hyde_retrieval(self, query: str) -> Dict:
        """HyDE: Generate hypothetical document"""
        # In production: use LLM to generate hypothetical answer
        hypothetical_doc = f"Hypothetical answer to: {query}"
        return {
            'strategy': 'hyde',
            'query': query,
            'hypothetical_doc': hypothetical_doc,
            'results': []
        }
    
    def _graph_retrieval(self, query: str) -> Dict:
        """Graph-based retrieval"""
        return {'strategy': 'graph', 'query': query, 'graph_results': []}
    
    def _multi_hop_retrieval(self, query: str) -> Dict:
        """Multi-step retrieval"""
        # Break query into sub-queries
        sub_queries = [f"Sub-query 1 of: {query}"]
        return {
            'strategy': 'multi_hop',
            'query': query,
            'sub_queries': sub_queries,
            'results': []
        }
    
    def query(self, question: str) -> Dict:
        """
        Complete agentic RAG pipeline
        1. Analyze query
        2. Choose strategy
        3. Execute retrieval
        4. Return results
        """
        print(f"\nAgentic RAG Query: {question}")
        print("-" * 60)
        
        # Step 1: Analyze
        analysis = self.analyze_query(question)
        print(f"Analysis:")
        print(f"  Question type: {analysis['question_type']}")
        print(f"  Entities: {analysis['num_entities']}")
        print(f"  Complexity: {analysis['complexity']}")
        
        # Step 2: Choose strategy
        strategy = analysis['recommended_strategy']
        print(f"  Recommended strategy: {strategy.value}")
        
        # Step 3: Execute
        results = self.execute_retrieval(question, strategy)
        
        # Step 4: Return
        return {
            'analysis': analysis,
            'retrieval_results': results
        }


# Example usage
agent = RetrievalAgent()

test_queries = [
    "What is machine learning?",
    "How does reinforcement learning work?",
    "Compare BERT and GPT transformers",
    "Why is deep learning important for computer vision?"
]

print("Agentic RAG Examples")
print("=" * 60)

for query in test_queries:
    results = agent.query(query)
    print(f"\nStrategy used: {results['retrieval_results']['strategy']}")
    print("-" * 60)

## Part 3: HyDE - Hypothetical Document Embeddings

In [ ]:
class HyDERetriever:
    """
    HyDE: Hypothetical Document Embeddings
    
    Key idea:
    1. Generate hypothetical answer to query
    2. Embed the hypothetical answer
    3. Use it for retrieval (bridges query-document gap)
    
    Benefits:
    - Better semantic matching
    - Handles complex queries
    - Improves retrieval quality
    """
    
    def __init__(self, llm, embedder, vector_store):
        self.llm = llm
        self.embedder = embedder
        self.vector_store = vector_store
    
    def retrieve(self, query: str, top_k: int = 5) -> Dict:
        """
        HyDE retrieval pipeline
        """
        # Step 1: Generate hypothetical document
        hypothetical_doc = self._generate_hypothetical(query)
        
        # Step 2: Embed hypothetical document
        hypothetical_embedding = self.embedder.embed_query(hypothetical_doc)
        
        # Step 3: Retrieve using hypothetical embedding
        # In production: results = vector_store.similarity_search_by_vector(
        #     hypothetical_embedding, k=top_k
        # )
        results = []  # Placeholder
        
        return {
            'query': query,
            'hypothetical_doc': hypothetical_doc,
            'retrieved_documents': results
        }
    
    def _generate_hypothetical(self, query: str) -> str:
        """Generate hypothetical answer"""
        # In production: use LLM
        prompt = f"""Write a detailed, informative answer to this question.
The answer should be factual and comprehensive.

Question: {query}

Answer:"""
        
        # Mock response
        return f"Hypothetical detailed answer to: {query}"


# Example
print("HyDE Example")
print("=" * 60)

class MockLLM:
    def generate(self, prompt):
        return "Generated text"

hyde = HyDERetriever(MockLLM(), MockEmbedder(), None)

query = "What are the key differences between supervised and unsupervised learning?"
results = hyde.retrieve(query)

print(f"Query: {query}")
print(f"\nHypothetical document: {results['hypothetical_doc'][:200]}...")

## Part 4: Parent Document Retriever - Hierarchical Retrieval

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Document:
    """Document with parent-child relationships"""
    id: str
    content: str
    metadata: Dict = field(default_factory=dict)
    parent_id: str = None
    children: List[str] = field(default_factory=list)


class ParentDocumentRetriever:
    """
    Parent Document Retriever
    
    Strategy:
    1. Split documents into small chunks (for embedding)
    2. Keep track of parent documents
    3. Retrieve small chunks
    4. Return larger parent context for better answers
    
    Benefits:
    - Precise retrieval (small chunks)
    - Rich context (parent documents)
    - Better LLM answers
    """
    
    def __init__(self, chunk_size: int = 200, parent_size: int = 2000):
        self.chunk_size = chunk_size
        self.parent_size = parent_size
        self.documents = {}  # id -> Document
        self.chunks = {}     # chunk_id -> Document
        self.parent_map = {} # chunk_id -> parent_id
    
    def add_document(self, doc: Document):
        """Add document and create chunks"""
        self.documents[doc.id] = doc
        
        # Create chunks
        chunks = self._create_chunks(doc)
        
        for chunk in chunks:
            self.chunks[chunk.id] = chunk
            self.parent_map[chunk.id] = doc.id
            doc.children.append(chunk.id)
    
    def _create_chunks(self, doc: Document) -> List[Document]:
        """Split document into chunks"""
        words = doc.content.split()
        chunks = []
        
        for i in range(0, len(words), self.chunk_size):
            chunk_content = ' '.join(words[i:i + self.chunk_size])
            chunk = Document(
                id=f"{doc.id}_chunk_{i}",
                content=chunk_content,
                metadata=doc.metadata.copy(),
                parent_id=doc.id
            )
            chunks.append(chunk)
        
        return chunks
    
    def retrieve(self, query: str, top_k: int = 5) -> Dict:
        """
        Retrieve with parent document context
        
        1. Find relevant chunks
        2. Get parent documents
        3. Return parent context
        """
        # In production: search chunks by embedding
        # chunk_results = vector_store.similarity_search(query, k=top_k * 2)
        
        # Mock: use all chunks
        chunk_results = list(self.chunks.values())[:top_k]
        
        # Get unique parent IDs
        parent_ids = set()
        for chunk in chunk_results:
            if chunk.id in self.parent_map:
                parent_ids.add(self.parent_map[chunk.id])
        
        # Retrieve parent documents
        parent_docs = [self.documents[pid] for pid in parent_ids if pid in self.documents]
        
        return {
            'query': query,
            'chunk_results': chunk_results,
            'parent_documents': parent_docs,
            'context': self._build_context(parent_docs)
        }
    
    def _build_context(self, parent_docs: List[Document]) -> str:
        """Build context from parent documents"""
        contexts = []
        for doc in parent_docs:
            context = f"[Document: {doc.id}]\n{doc.content[:500]}..."
            contexts.append(context)
        
        return "\n\n".join(contexts)


# Example
print("Parent Document Retriever Example")
print("=" * 60)

retriever = ParentDocumentRetriever(chunk_size=100, parent_size=1000)

# Add sample documents
docs = [
    Document(
        id="doc1",
        content="Machine learning is a subset of artificial intelligence. " * 20,
        metadata={'source': 'ML Basics'}
    ),
    Document(
        id="doc2",
        content="Deep learning uses neural networks with multiple layers. " * 20,
        metadata={'source': 'DL Fundamentals'}
    )
]

for doc in docs:
    retriever.add_document(doc)

print(f"Added {len(retriever.documents)} documents")
print(f"Created {len(retriever.chunks)} chunks")

# Retrieve
results = retriever.retrieve("What is machine learning?")
print(f"\nRetrieved {len(results['chunk_results'])} chunks")
print(f"Returning {len(results['parent_documents'])} parent documents")
print(f"\nContext preview: {results['context'][:300]}...")

## Summary: Advanced RAG Patterns

In [ ]:
from IPython.display import Markdown

summary = """
## Advanced RAG Patterns - Summary

### 1. Graph RAG
- **What**: Combines vector search with knowledge graphs
- **When**: Multi-hop reasoning, relationship queries
- **Benefits**: Better for complex relationship questions
- **Implementation**: Build KG, query both vector and graph, combine results

### 2. Agentic RAG
- **What**: AI agent decides retrieval strategy
- **When**: Diverse query types, complex information needs
- **Benefits**: Adaptive retrieval, better resource utilization
- **Strategies**: Direct, Expanded, HyDE, Graph, Multi-hop

### 3. HyDE (Hypothetical Document Embeddings)
- **What**: Generate hypothetical answer, embed it, retrieve
- **When**: Complex queries, query-document gap
- **Benefits**: Better semantic matching
- **Implementation**: LLM generates hypothetical doc, embed, retrieve

### 4. Parent Document Retriever
- **What**: Retrieve small chunks, return large parent context
- **When**: Need precise retrieval + rich context
- **Benefits**: Better retrieval accuracy + better LLM answers
- **Implementation**: Chunk documents, track parents, retrieve chunks, return parents

### When to Use Each Pattern

| Pattern | Best For | Complexity |
|---------|----------|------------|
| **Basic RAG** | Simple Q&A | Low |
| **Graph RAG** | Relationship queries | High |
| **Agentic RAG** | Diverse queries | Medium |
| **HyDE** | Complex questions | Medium |
| **Parent Document** | Long documents | Low |

### Next Steps
1. Implement with real LLM and vector store
2. Combine multiple patterns
3. Evaluate performance
4. Optimize for your use case
"""

Markdown(summary)